# MAGDA Fine-Tuning: Qwen 2.5 Coder 7B

Fine-tunes a LoRA adapter for three tasks:
- **Router**: classify intent (COMMAND / MUSIC / BOTH)
- **Command**: generate MAGDA DSL code for DAW operations
- **Music**: generate chord progressions, notes, arpeggios in compact notation

Requirements: Colab Pro (A100 GPU recommended, T4 works too)

In [ ]:
# Step 1: Install dependencies
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# Step 2: Upload dataset.jsonl
# Option A: Upload from local machine
from google.colab import files
uploaded = files.upload()  # select dataset.jsonl

# Option B: If using Google Drive, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/magda/dataset.jsonl .

In [ ]:
# Step 3: Load and inspect dataset
import json
from datasets import Dataset

with open('dataset.jsonl', 'r') as f:
    raw_data = [json.loads(line) for line in f]

print(f"Total examples: {len(raw_data)}")
print(f"Example: {json.dumps(raw_data[0], indent=2)[:300]}")

In [ ]:
# Step 4: Load base model with unsloth (4-bit quantized for training)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=2048,
    dtype=None,       # auto-detect
    load_in_4bit=True,
)

print(f"Model loaded: {model.config._name_or_path}")

In [ ]:
# Step 5: Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                    # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,          # optimized = 0
    bias="none",
    use_gradient_checkpointing="unsloth",  # saves VRAM
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
# Step 6: Format dataset for Qwen chat template

def format_chat(example):
    """Convert messages list to Qwen ChatML format."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chat)

# Preview a formatted example
print(dataset[0]["text"][:500])
print("---")
print(dataset[-1]["text"][:500])

In [ ]:
# Step 7: Train
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=True,  # pack short examples together for efficiency
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not __import__('torch').cuda.is_bf16_supported(),
        bf16=__import__('torch').cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="magda-lora-output",
        report_to="none",
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"Training complete! Loss: {stats.training_loss:.4f}")

In [ ]:
# Step 8: Quick validation — test all three tasks
# Use Unsloth's native inference mode (avoids RoPE shape bugs with merge_and_unload)
from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_inference(model)

COMMAND_SYSTEM = (
    "You are MAGDA, a DAW AI assistant. Respond ONLY with DSL code. No prose.\n"
    "Syntax: track(name=\"X\", new=true), track(id=N), filter(tracks, track.name == \"X\")\n"
    "Chains: .clip.new(bar=1, length_bars=4), .track.set(volume_db=-6, pan=0.5, mute=true, solo=true),\n"
    ".fx.add(name=\"reverb\"), .delete(), .select(), .clip.rename(index=0, name=\"X\"),\n"
    ".clip.delete(index=0), .for_each(...), .clips.select(clip.length_bars > 2)\n"
    "Notes: .notes.add(pitch=C4, beat=0, length=1, velocity=100),\n"
    ".notes.add_chord(root=C4, quality=major, beat=0, length=1),\n"
    ".notes.add_arpeggio(root=C4, quality=major, beat=0, step=0.5, pattern=up, fill=true)\n"
    "Functions: random(min, max)\n"
    "When user says 'create a track', use new=true. One statement per line."
)

test_cases = [
    # Router
    {"messages": [
        {"role": "system", "content": "Classify the user's request as COMMAND, MUSIC, or BOTH. Respond with ONLY one word."},
        {"role": "user", "content": "create a track called Synth Lead and add a chord progression"},
    ]},
    # Command (no state)
    {"messages": [
        {"role": "system", "content": COMMAND_SYSTEM},
        {"role": "user", "content": "create a track called Horns, add reverb, and a 4 bar clip at bar 1"},
    ]},
    # Command (with state)
    {"messages": [
        {"role": "system", "content": COMMAND_SYSTEM + "\n\nCurrent DAW state:\n{\"tracks\":[{\"id\":1,\"name\":\"Piano\",\"type\":\"Audio\"}],\"track_count\":1,\"selected_track_id\":1}"},
        {"role": "user", "content": "add an 8 bar clip and set volume to -6"},
    ]},
    # Music (compact)
    {"messages": [
        {"role": "system", "content": "Generate musical content using compact notation. Respond ONLY with instructions. No prose. One instruction per line."},
        {"role": "user", "content": "ii-V-I in Ab major"},
    ]},
]

for i, tc in enumerate(test_cases):
    text = tokenizer.apply_chat_template(
        tc["messages"],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
        )
    
    result = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    print(f"\n--- Test {i+1}: {tc['messages'][-1]['content']} ---")
    print(result.strip())

In [ ]:
# Step 9: Save LoRA adapter
model.save_pretrained("magda-lora")
tokenizer.save_pretrained("magda-lora")
print("LoRA adapter saved to magda-lora/")

In [ ]:
# Step 10: Merge LoRA into base model and save as full model
# This creates the merged 16-bit model needed for GGUF export
model.save_pretrained_merged(
    "magda-merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Merged model saved to magda-merged/")

In [ ]:
# Step 11: Export to GGUF Q4_K_M
# This creates the quantized model file you can run with llama-server
model.save_pretrained_gguf(
    "magda-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF exported to magda-gguf/")

In [ ]:
# Step 12: Download the GGUF file
import glob
gguf_files = glob.glob("magda-gguf/*.gguf")
print(f"GGUF files: {gguf_files}")

if gguf_files:
    from google.colab import files
    files.download(gguf_files[0])
    print(f"Downloading {gguf_files[0]}...")
else:
    print("No GGUF file found. Check the magda-gguf/ directory.")
    !ls -la magda-gguf/

## Done!

Copy the downloaded `.gguf` file to your models directory and point MAGDA at it:

```
cp magda-merged-q4_k_m.gguf /Volumes/External\ SSD/models/magda/
```

Then update the model path in MAGDA preferences.